# BART Ridership Data Audit

This notebook audits the processed ridership pipeline used by the Dash app. The key data decision is:

```text
Hourly OD is the canonical ridership source where available.
Workbook Total Trips / Total Trips OD files are validation benchmarks and fallback sources.
```

The notebook focuses on station-code normalization, source coverage, and differences between hourly OD and workbook totals.


In [1]:
from pathlib import Path
import sys

import pandas as pd

pd.options.display.float_format = "{:,.2f}".format
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)


def find_project_root(start=None):
    """Find the repo root from either notebooks/ or the repo root itself."""
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing src/ and data/.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REFERENCE = PROJECT_ROOT / "data" / "reference"
PROCESSED = PROJECT_ROOT / "data" / "processed"
RAW = PROJECT_ROOT / "data" / "raw"
HOURLY_OD_DIR = RAW / "Hourly_OD"

PROJECT_ROOT


WindowsPath('c:/Users/Carl Wheezer/Documents/git-test/bart-ridership-dash')

## Reference and Processed Data Inputs

The app should read stable reference tables and processed files. Raw hourly OD CSVs and workbooks are used by `scripts/prepare_data.py` to regenerate processed artifacts and validation outputs.


In [2]:
station_master_path = REFERENCE / "stations_master.csv"
station_aliases_path = REFERENCE / "station_aliases.csv"
app_summary_path = PROCESSED / "station_ridership_monthly_summary.csv"
hourly_summary_path = PROCESSED / "hourly_od_station_monthly_summary.csv"
validation_path = PROCESSED / "hourly_od_total_trips_validation.csv"
hourly_quality_path = PROCESSED / "hourly_od_completeness_audit.csv"

input_files_df = pd.DataFrame(
    [
        {
            "Artifact": path.name,
            "Role": role,
            "Path": path.relative_to(PROJECT_ROOT).as_posix(),
            "Exists": path.exists(),
            "Size MB": path.stat().st_size / 1_000_000 if path.exists() else None,
        }
        for role, path in [
            ("Station identity reference", station_master_path),
            ("Raw-code alias reference", station_aliases_path),
            ("App monthly station summary", app_summary_path),
            ("Hourly OD monthly station summary", hourly_summary_path),
            ("Hourly OD vs workbook validation", validation_path),
            ("Hourly OD completeness and imputation audit", hourly_quality_path),
        ]
    ]
)
input_files_df


,Artifact,Role,Path,Exists,Size MB
0,stations_master.csv,Station identity reference,data/reference/stations_master.csv,True,0.00
1,station_aliases.csv,Raw-code alias reference,data/reference/station_aliases.csv,True,0.01
2,station_ridership_monthly_summary.csv,App monthly station summary,data/processed/station_ridership_monthly_summa...,True,0.86
3,hourly_od_station_monthly_summary.csv,Hourly OD monthly station summary,data/processed/hourly_od_station_monthly_summa...,True,0.86
4,hourly_od_total_trips_validation.csv,Hourly OD vs workbook validation,data/processed/hourly_od_total_trips_validatio...,True,1.44
5,hourly_od_completeness_audit.csv,Hourly OD completeness and imputation audit,data/processed/hourly_od_completeness_audit.csv,True,0.02


In [3]:
station_master_df = pd.read_csv(station_master_path, dtype={"station_id": "string"})
station_aliases_df = pd.read_csv(station_aliases_path, dtype={"source_system": "string", "raw_code": "string", "station_id": "string"})
app_summary_df = pd.read_csv(app_summary_path)
hourly_summary_df = pd.read_csv(hourly_summary_path)
valid_ridership_df = pd.read_csv(validation_path)
hourly_quality_df = pd.read_csv(hourly_quality_path)

summary_shapes_df = pd.DataFrame(
    [
        {"Table": "Station master reference", "Rows": len(station_master_df), "Columns": len(station_master_df.columns)},
        {"Table": "Station alias reference", "Rows": len(station_aliases_df), "Columns": len(station_aliases_df.columns)},
        {"Table": "App monthly station summary", "Rows": len(app_summary_df), "Columns": len(app_summary_df.columns)},
        {"Table": "Hourly OD monthly station summary", "Rows": len(hourly_summary_df), "Columns": len(hourly_summary_df.columns)},
        {"Table": "Hourly OD vs workbook validation", "Rows": len(valid_ridership_df), "Columns": len(valid_ridership_df.columns)},
        {"Table": "Hourly OD completeness and imputation audit", "Rows": len(hourly_quality_df), "Columns": len(hourly_quality_df.columns)},
    ]
)
summary_shapes_df


,Table,Rows,Columns
0,Station master reference,50,7
1,Station alias reference,201,3
2,App monthly station summary,4755,10
3,Hourly OD monthly station summary,4755,10
4,Hourly OD vs workbook validation,4741,14
5,Hourly OD completeness and imputation audit,96,20


In [4]:
period_coverage_df = (
    app_summary_df
    .groupby(["Year", "Month", "Period", "Source Type"], as_index=False)
    .agg(
        Stations=("Entry Station", "nunique"),
        Total_Ridership=("Ridership", "sum"),
    )
    .sort_values(["Year", "Month"])
)

period_coverage_display_df = period_coverage_df[
    ["Period", "Source Type", "Stations", "Total_Ridership"]
]

pd.concat([period_coverage_display_df.head(), period_coverage_display_df.tail()])


,Period,Source Type,Stations,Total_Ridership
0,2018-01,hourly_od,46,"9,785,965.00"
1,2018-02,hourly_od,46,"9,262,187.00"
2,2018-03,hourly_od,47,"10,364,615.00"
3,2018-04,hourly_od,48,"10,049,472.00"
4,2018-05,hourly_od,48,"10,595,609.00"
91,2025-08,hourly_od,50,"4,908,797.00"
92,2025-09,hourly_od,50,"4,724,313.00"
93,2025-10,hourly_od,50,"4,930,734.00"
94,2025-11,hourly_od,50,"4,031,002.00"
95,2025-12,hourly_od,50,"3,768,928.00"


In [5]:
source_type_summary_df = (
    app_summary_df
    .groupby("Source Type", as_index=False)
    .agg(
        Rows=("Entry Station", "count"),
        Periods=("Period", "nunique"),
        Total_Ridership=("Ridership", "sum"),
    )
)
source_type_summary_df


,Source Type,Rows,Periods,Total_Ridership
0,hourly_od,4755,96,"494,157,915.41"


## January 2018 Workbook Edge Case

`Ridership_201801.xlsx` does not contain a comparable monthly Total Trips sheet. It contains average weekday/Saturday/Sunday OD sheets, so January 2018 is taken from hourly OD instead.


In [6]:
jan_2018_workbook = RAW / "ridership_2018" / "Ridership_201801.xlsx"
jan_2018_hourly = HOURLY_OD_DIR / "date-hour-soo-dest-2018.csv"

jan_2018_sheets = pd.ExcelFile(jan_2018_workbook).sheet_names
jan_2018_sheets


['Weekday OD', 'Saturday OD', 'Sunday OD']

In [7]:
def hourly_month_total(csv_path, year, month, chunksize=500_000):
    """Compute one monthly total without loading the full hourly CSV into memory."""
    period_prefix = f"{year}-{month:02d}"
    total = 0
    for chunk in pd.read_csv(
        csv_path,
        header=None,
        names=["Date", "Hour", "Origin", "Destination", "Exits"],
        usecols=["Date", "Exits"],
        chunksize=chunksize,
    ):
        selected = chunk[chunk["Date"].astype(str).str.startswith(period_prefix)]
        total += pd.to_numeric(selected["Exits"], errors="coerce").fillna(0).sum()
    return int(total)

jan_2018_hourly_total = hourly_month_total(jan_2018_hourly, 2018, 1)
jan_2018_app_total = int(
    app_summary_df.loc[(app_summary_df["Year"] == 2018) & (app_summary_df["Month"] == 1), "Ridership"].sum()
)

pd.DataFrame(
    [
        {"Metric": "January 2018 hourly OD total", "Ridership": jan_2018_hourly_total},
        {"Metric": "January 2018 app summary total", "Ridership": jan_2018_app_total},
    ]
)


,Metric,Ridership
0,January 2018 hourly OD total,9785965
1,January 2018 app summary total,9785965


## Station Alias Audit

`stations_master.csv` is the clean station dimension table. `station_aliases.csv` is the compact machine-readable crosswalk from raw source codes to official `station_id` values. This section enriches that alias table for human audit purposes without pushing noisy source details back into the master table.


In [8]:
legacy_app_to_station_id = (
    station_aliases_df[station_aliases_df["source_system"] == "legacy_app_code"]
    .set_index("raw_code")["station_id"]
)

alias_audit_df = station_aliases_df.merge(
    station_master_df[["station_id", "display_name", "opened_date", "closed_date"]],
    on="station_id",
    how="left",
    validate="many_to_one",
)
alias_audit_df["raw_code_matches_station_id"] = (
    alias_audit_df["raw_code"].astype(str).str.lower()
    == alias_audit_df["station_id"].astype(str).str.lower()
)
alias_audit_df = alias_audit_df[
    [
        "source_system",
        "raw_code",
        "station_id",
        "display_name",
        "opened_date",
        "closed_date",
        "raw_code_matches_station_id",
    ]
].sort_values(["source_system", "station_id", "raw_code"]).reset_index(drop=True)

alias_display_df = alias_audit_df[
    ["source_system", "raw_code", "station_id", "display_name", "raw_code_matches_station_id"]
]

alias_display_df


,source_system,raw_code,station_id,display_name,raw_code_matches_station_id
0,bart_deprecated_abbr,ASBY,ashb,Ashby,False
1,bart_official_abbr,12TH,12th,12th St/Oakland City Center,True
2,bart_official_abbr,16TH,16th,16th St/Mission,True
3,bart_official_abbr,19TH,19th,19th St/Oakland,True
4,bart_official_abbr,24TH,24th,24th St/Mission,True
...,...,...,...,...,...
196,workbook_total_trips,UC,ucty,Union City,False
197,workbook_total_trips,WS,warm,Warm Springs/South Fremont,False
198,workbook_total_trips,WC,wcrk,Walnut Creek,False
199,workbook_total_trips,WD,wdub,West Dublin/Pleasanton,False


In [9]:
# If this table is non-empty, an alias points to a station_id missing from stations_master.csv.
alias_target_holes_df = alias_audit_df[alias_audit_df["display_name"].isna()].copy()

alias_target_holes_df


,source_system,raw_code,station_id,display_name,opened_date,closed_date,raw_code_matches_station_id


In [10]:
alias_changes_only_df = alias_audit_df[~alias_audit_df["raw_code_matches_station_id"]].copy()

alias_source_summary_df = (
    alias_audit_df.groupby("source_system", as_index=False)
    .agg(
        alias_rows=("raw_code", "count"),
        covered_station_ids=("station_id", "nunique"),
        aliases_that_change_station_id=("raw_code_matches_station_id", lambda values: int((~values).sum())),
    )
    .sort_values("source_system")
)

alias_source_summary_df


,source_system,alias_rows,covered_station_ids,aliases_that_change_station_id
0,bart_deprecated_abbr,1,1,1
1,bart_official_abbr,50,50,0
2,hourly_od,50,50,0
3,legacy_app_code,50,50,45
4,workbook_total_trips,50,50,50


## Raw Hourly OD Code Coverage

This checks raw hourly OD station codes found in each yearly CSV against the `hourly_od` rows in `station_aliases.csv`. Any `Unmapped Raw Codes` value should be investigated before treating hourly OD as canonical for that year.


In [11]:
def raw_hourly_codes_for_file(csv_path, chunksize=500_000):
    codes = set()
    for chunk in pd.read_csv(
        csv_path,
        header=None,
        names=["Date", "Hour", "Origin", "Destination", "Exits"],
        usecols=["Origin", "Destination"],
        chunksize=chunksize,
    ):
        codes.update(chunk["Origin"].dropna().astype(str).str.strip().str.upper().unique())
        codes.update(chunk["Destination"].dropna().astype(str).str.strip().str.upper().unique())
    return codes


known_hourly_codes = set(
    station_aliases_df.loc[
        station_aliases_df["source_system"] == "hourly_od",
        "raw_code",
    ].astype(str)
)
hourly_code_coverage_rows = []
for csv_path in sorted(HOURLY_OD_DIR.glob("date-hour-soo-dest-*.csv")):
    year = int(csv_path.stem.split("-")[-1])
    raw_codes = raw_hourly_codes_for_file(csv_path)
    hourly_code_coverage_rows.append(
        {
            "Year": year,
            "Raw Code Count": len(raw_codes),
            "Unmapped Raw Codes": sorted(raw_codes - known_hourly_codes),
            "Mapped Raw Codes": sorted(raw_codes & known_hourly_codes),
        }
    )

hourly_code_coverage_df = pd.DataFrame(hourly_code_coverage_rows)
hourly_code_coverage_df[["Year", "Raw Code Count", "Unmapped Raw Codes"]]


,Year,Raw Code Count,Unmapped Raw Codes
0,2018,48,[]
1,2019,50,[]
2,2020,50,[]
3,2021,50,[]
4,2022,50,[]
5,2023,50,[]
6,2024,50,[]
7,2025,50,[]


## Hourly OD vs Workbook Validation

The validation table compares station-level monthly totals from hourly OD against workbook Total Trips / Total Trips OD. Since hourly OD is canonical where available, workbook values are treated as a benchmark rather than the runtime source of truth.


In [12]:
validation_audit_df = valid_ridership_df.copy()

if "station_id" not in validation_audit_df.columns:
    validation_audit_df["station_id"] = validation_audit_df["Entry Station"].astype(str).map(legacy_app_to_station_id)

# Normalize display_name defensively so this cell works with both freshly
# regenerated processed files and stale pre-refactor notebook state.
if "display_name" not in validation_audit_df.columns:
    validation_audit_df = validation_audit_df.merge(
        station_master_df[["station_id", "display_name"]],
        on="station_id",
        how="left",
        validate="many_to_one",
    )
else:
    validation_audit_df = validation_audit_df.merge(
        station_master_df[["station_id", "display_name"]].rename(
            columns={"display_name": "display_name_master"}
        ),
        on="station_id",
        how="left",
        validate="many_to_one",
    )
    validation_audit_df["display_name"] = validation_audit_df["display_name"].fillna(
        validation_audit_df["display_name_master"]
    )
    validation_audit_df = validation_audit_df.drop(columns=["display_name_master"])

validation_audit_df["expected_station_id"] = validation_audit_df["Entry Station"].astype(str).map(legacy_app_to_station_id)
validation_audit_df["Station ID Matches Alias"] = validation_audit_df["station_id"] == validation_audit_df["expected_station_id"]
validation_audit_df["Workbook Ridership"] = pd.to_numeric(validation_audit_df["Workbook Ridership"], errors="coerce").fillna(0)
validation_audit_df["Hourly OD Ridership"] = pd.to_numeric(validation_audit_df["Hourly OD Ridership"], errors="coerce").fillna(0)
validation_audit_df["Difference"] = validation_audit_df["Hourly OD Ridership"] - validation_audit_df["Workbook Ridership"]
validation_audit_df["Absolute Difference"] = validation_audit_df["Difference"].abs()
validation_audit_df["Percent Difference vs Workbook"] = (
    validation_audit_df["Difference"]
    / validation_audit_df["Workbook Ridership"].replace(0, pd.NA)
    * 100
)
validation_audit_df["Absolute Percent Difference vs Workbook"] = validation_audit_df["Percent Difference vs Workbook"].abs()
validation_audit_df["Difference Direction"] = validation_audit_df["Difference"].map(
    lambda value: "Hourly higher" if value > 0 else ("Workbook higher" if value < 0 else "Match")
)


def audit_flag(row):
    if pd.isna(row["station_id"]):
        return "Missing processed station_id"
    if pd.isna(row["display_name"]):
        return "Missing processed display_name"
    if not row["Station ID Matches Alias"]:
        return "Processed station_id does not match alias table"
    if row["Workbook Ridership"] == 0 and row["Hourly OD Ridership"] > 0:
        return "Missing workbook benchmark / workbook station code issue"
    if row["Hourly OD Ridership"] == 0 and row["Workbook Ridership"] > 0:
        return "Missing hourly OD benchmark / hourly station alias issue"
    if row["Absolute Difference"] >= 10_000:
        return "Large absolute difference"
    if pd.notna(row["Absolute Percent Difference vs Workbook"]) and row["Absolute Percent Difference vs Workbook"] >= 5:
        return "Large percentage difference"
    if row["Absolute Difference"] > 0:
        return "Small source-definition difference"
    return "Exact match"


validation_audit_df["Audit Flag"] = validation_audit_df.apply(audit_flag, axis=1)
validation_audit_df = validation_audit_df.sort_values(
    ["Audit Flag", "Absolute Difference"],
    ascending=[True, False],
).reset_index(drop=True)

validation_display_df = validation_audit_df[
    [
        "Period",
        "station_id",
        "display_name",
        "Workbook Ridership",
        "Hourly OD Ridership",
        "Difference",
        "Absolute Difference",
        "Percent Difference vs Workbook",
        "Audit Flag",
    ]
]

validation_display_df


,Period,station_id,display_name,Workbook Ridership,Hourly OD Ridership,Difference,Absolute Difference,Percent Difference vs Workbook,Audit Flag
0,2018-05,bery,Berryessa/North San Jose,0.00,0.00,0.00,0.00,<NA>,Exact match
1,2018-05,mlpt,Milpitas,0.00,0.00,0.00,0.00,<NA>,Exact match
2,2018-06,bery,Berryessa/North San Jose,0.00,0.00,0.00,0.00,<NA>,Exact match
3,2018-06,mlpt,Milpitas,0.00,0.00,0.00,0.00,<NA>,Exact match
4,2018-07,bery,Berryessa/North San Jose,0.00,0.00,0.00,0.00,<NA>,Exact match
...,...,...,...,...,...,...,...,...,...
4736,2020-04,oakl,Oakland International Airport,961.00,979.00,18.00,18.00,1.87,Small source-definition difference
4737,2020-02,hayw,Hayward,"105,688.00","105,672.33",-15.67,15.67,-0.01,Small source-definition difference
4738,2025-03,oakl,Oakland International Airport,"17,448.00","17,436.00",-12.00,12.00,-0.07,Small source-definition difference
4739,2020-05,oakl,Oakland International Airport,"1,626.00","1,635.00",9.00,9.00,0.55,Small source-definition difference


In [13]:
audit_flag_summary_df = (
    validation_audit_df
    .groupby("Audit Flag", as_index=False)
    .agg(
        Rows=("Period", "count"),
        Periods=("Period", "nunique"),
        Stations=("station_id", "nunique"),
        Max_Absolute_Difference=("Absolute Difference", "max"),
        Mean_Absolute_Difference=("Absolute Difference", "mean"),
    )
    .sort_values("Rows", ascending=False)
)
audit_flag_summary_df


,Audit Flag,Rows,Periods,Stations,Max_Absolute_Difference,Mean_Absolute_Difference
4,Small source-definition difference,4204,94,50,"9,949.00","1,336.78"
2,Large percentage difference,372,71,44,"9,951.00","4,008.71"
1,Large absolute difference,112,59,18,"64,828.00","17,755.51"
0,Exact match,32,17,2,0.00,0.00
3,Missing workbook benchmark / workbook station ...,21,12,4,138.00,41.90


In [14]:
priority_validation_rows_df = validation_audit_df[
    validation_audit_df["Audit Flag"].isin(
        [
            "Missing workbook benchmark / workbook station code issue",
            "Missing hourly OD benchmark / hourly station alias issue",
            "Large absolute difference",
            "Large percentage difference",
        ]
    )
].sort_values("Absolute Difference", ascending=False)

priority_validation_display_df = priority_validation_rows_df[
    [
        "Period",
        "station_id",
        "display_name",
        "Workbook Ridership",
        "Hourly OD Ridership",
        "Difference",
        "Absolute Difference",
        "Percent Difference vs Workbook",
        "Audit Flag",
    ]
]

priority_validation_display_df


,Period,station_id,display_name,Workbook Ridership,Hourly OD Ridership,Difference,Absolute Difference,Percent Difference vs Workbook,Audit Flag
32,2025-12,embr,Embarcadero,"373,590.00","308,762.00","-64,828.00","64,828.00",-17.35,Large absolute difference
33,2025-12,powl,Powell St,"352,353.00","291,256.00","-61,097.00","61,097.00",-17.34,Large absolute difference
34,2025-12,mont,Montgomery St,"303,035.00","254,156.00","-48,879.00","48,879.00",-16.13,Large absolute difference
35,2025-10,embr,Embarcadero,"464,508.00","419,759.00","-44,749.00","44,749.00",-9.63,Large absolute difference
36,2025-10,powl,Powell St,"390,272.00","349,007.00","-41,265.00","41,265.00",-10.57,Large absolute difference
...,...,...,...,...,...,...,...,...,...
532,2018-03,antc,Antioch,0.00,4.00,4.00,4.00,<NA>,Missing workbook benchmark / workbook station ...
533,2020-02,bery,Berryessa/North San Jose,0.00,3.00,3.00,3.00,<NA>,Missing workbook benchmark / workbook station ...
534,2020-04,mlpt,Milpitas,0.00,3.00,3.00,3.00,<NA>,Missing workbook benchmark / workbook station ...
535,2019-12,bery,Berryessa/North San Jose,0.00,2.00,2.00,2.00,<NA>,Missing workbook benchmark / workbook station ...


In [15]:
station_validation_summary_df = (
    validation_audit_df
    .groupby(["station_id", "display_name"], dropna=False)
    .agg(
        Periods_Compared=("Period", "nunique"),
        Rows_With_Difference=("Has Difference", "sum"),
        Max_Absolute_Difference=("Absolute Difference", "max"),
        Mean_Absolute_Difference=("Absolute Difference", "mean"),
        Total_Absolute_Difference=("Absolute Difference", "sum"),
        Max_Absolute_Percent_Difference=("Absolute Percent Difference vs Workbook", "max"),
        Missing_Hourly_Rows=("Audit Flag", lambda values: (values == "Missing hourly OD benchmark / hourly station alias issue").sum()),
        Missing_Workbook_Rows=("Audit Flag", lambda values: (values == "Missing workbook benchmark / workbook station code issue").sum()),
        Large_Absolute_Difference_Rows=("Audit Flag", lambda values: (values == "Large absolute difference").sum()),
        Large_Percentage_Difference_Rows=("Audit Flag", lambda values: (values == "Large percentage difference").sum()),
    )
    .reset_index()
    .sort_values(["Missing_Hourly_Rows", "Missing_Workbook_Rows", "Max_Absolute_Difference"], ascending=False)
)

station_validation_display_df = station_validation_summary_df[
    [
        "station_id",
        "display_name",
        "Periods_Compared",
        "Rows_With_Difference",
        "Max_Absolute_Difference",
        "Mean_Absolute_Difference",
        "Total_Absolute_Difference",
        "Max_Absolute_Percent_Difference",
        "Large_Absolute_Difference_Rows",
        "Large_Percentage_Difference_Rows",
    ]
]

station_validation_display_df


,station_id,display_name,Periods_Compared,Rows_With_Difference,Max_Absolute_Difference,Mean_Absolute_Difference,Total_Absolute_Difference,Max_Absolute_Percent_Difference,Large_Absolute_Difference_Rows,Large_Percentage_Difference_Rows
8,bery,Berryessa/North San Jose,92,77,"7,114.00","1,050.34","96,631.00",27.96,0,32
27,mlpt,Milpitas,92,75,"7,142.00",691.52,"63,620.00",53.53,0,15
4,antc,Antioch,94,94,"5,251.00","1,711.10","160,843.50",11.91,0,23
33,pctr,Pittsburg Center,93,93,"1,151.00",403.04,"37,482.33",9.70,0,3
18,embr,Embarcadero,95,95,"64,828.00","5,598.71","531,877.67",17.35,6,0
37,powl,Powell St,95,95,"61,097.00","5,599.35","531,938.17",17.34,4,0
28,mont,Montgomery St,95,95,"48,879.00","4,325.40","410,913.20",16.13,4,0
10,civc,Civic Center/UN Plaza,95,95,"35,712.00","5,165.68","490,739.75",13.99,4,0
42,sfia,San Francisco International Airport,95,95,"26,789.00","7,541.76","716,466.83",20.39,34,12
1,16th,16th St/Mission,95,95,"23,388.00","2,110.74","200,520.43",15.64,4,0


## Period-Level Source Differences

Period-level differences reveal months where the two sources diverge broadly, which is different from a single station-code mapping issue.


In [16]:
period_validation_summary_df = (
    validation_audit_df
    .groupby(["Year", "Month", "Period"], as_index=False)
    .agg(
        Workbook_Total=("Workbook Ridership", "sum"),
        Hourly_OD_Total=("Hourly OD Ridership", "sum"),
        Total_Absolute_Station_Difference=("Absolute Difference", "sum"),
        Max_Station_Difference=("Absolute Difference", "max"),
        Stations_Compared=("station_id", "nunique"),
        Priority_Row_Count=(
            "Audit Flag",
            lambda values: values.isin(
                [
                    "Missing workbook benchmark / workbook station code issue",
                    "Missing hourly OD benchmark / hourly station alias issue",
                    "Large absolute difference",
                    "Large percentage difference",
                ]
            ).sum(),
        ),
    )
)
period_validation_summary_df["Total Difference"] = period_validation_summary_df["Hourly_OD_Total"] - period_validation_summary_df["Workbook_Total"]
period_validation_summary_df["Total Percent Difference vs Workbook"] = (
    period_validation_summary_df["Total Difference"]
    / period_validation_summary_df["Workbook_Total"].replace(0, pd.NA)
    * 100
)
period_validation_summary_df = period_validation_summary_df.sort_values("Max_Station_Difference", ascending=False)

period_validation_display_df = period_validation_summary_df[
    [
        "Period",
        "Workbook_Total",
        "Hourly_OD_Total",
        "Total Difference",
        "Total Percent Difference vs Workbook",
        "Total_Absolute_Station_Difference",
        "Max_Station_Difference",
        "Stations_Compared",
        "Priority_Row_Count",
    ]
]

period_validation_display_df


,Period,Workbook_Total,Hourly_OD_Total,Total Difference,Total Percent Difference vs Workbook,Total_Absolute_Station_Difference,Max_Station_Difference,Stations_Compared,Priority_Row_Count
94,2025-12,"4,395,799.00","3,768,928.00","-626,871.00",-14.26,"626,871.00","64,828.00",50,50
92,2025-10,"5,346,891.00","4,930,734.00","-416,157.00",-7.78,"416,207.00","44,749.00",50,39
93,2025-11,"4,409,065.00","4,031,002.00","-378,063.00",-8.57,"378,063.00","39,498.00",50,41
91,2025-09,"5,047,121.00","4,724,313.00","-322,808.00",-6.40,"323,310.00","36,370.00",50,28
80,2024-10,"4,831,431.00","4,941,039.00","109,608.00",2.27,"109,608.00","20,724.00",50,4
83,2025-01,"4,151,331.00","4,261,168.00","109,837.00",2.65,"109,837.00","20,052.00",50,4
89,2025-07,"4,713,700.00","4,823,002.00","109,302.00",2.32,"109,302.00","19,901.00",50,5
82,2024-12,"3,874,228.00","3,964,739.00","90,511.00",2.34,"92,769.00","19,813.00",50,4
87,2025-05,"4,670,575.00","4,775,431.00","104,856.00",2.25,"104,856.00","19,629.00",50,5
86,2025-04,"4,736,423.00","4,846,643.00","110,220.00",2.33,"110,220.00","19,506.00",50,4


## Hourly OD Completeness Audit

This table is produced by `scripts/prepare_data.py` as `data/processed/hourly_od_completeness_audit.csv`. It documents missing full days that were imputed before monthly summaries were written and flags major source-definition discrepancies where hourly OD remains canonical but workbook totals are useful context.


In [17]:
hourly_completeness_audit_df = hourly_quality_df.copy()

hourly_completeness_display_df = hourly_completeness_audit_df[
    [
        "Period",
        "Quality Flag",
        "Calendar Days",
        "Observed Days",
        "Missing Full Days",
        "Missing Full Day List",
        "Min Present Hours",
        "Median Present Hours",
        "Raw Hourly Ridership",
        "Imputed Hourly Ridership",
        "Imputed Ridership Added",
        "Workbook Ridership",
        "Workbook Difference Pct",
    ]
].sort_values(["Quality Flag", "Period"]).reset_index(drop=True)

hourly_completeness_display_df


,Period,Quality Flag,Calendar Days,Observed Days,Missing Full Days,Missing Full Day List,Min Present Hours,Median Present Hours,Raw Hourly Ridership,Imputed Hourly Ridership,Imputed Ridership Added,Workbook Ridership,Workbook Difference Pct
0,2018-01,clean_or_minor_difference,31,31,0,NaN,20,23.00,"9,785,965.00","9,785,965.00",0.00,NaN,NaN
1,2018-02,clean_or_minor_difference,28,28,0,NaN,19,23.00,"9,262,187.00","9,262,187.00",0.00,"9,189,913.00",0.01
2,2018-03,clean_or_minor_difference,31,31,0,NaN,20,23.00,"10,364,615.00","10,364,615.00",0.00,"10,278,960.00",0.01
3,2018-04,clean_or_minor_difference,30,30,0,NaN,21,23.00,"10,049,472.00","10,049,472.00",0.00,"9,950,264.00",0.01
4,2018-05,clean_or_minor_difference,31,31,0,NaN,22,23.00,"10,595,609.00","10,595,609.00",0.00,"10,486,845.00",0.01
5,2018-06,clean_or_minor_difference,30,30,0,NaN,21,23.00,"10,432,529.00","10,432,529.00",0.00,"10,331,906.00",0.01
6,2018-07,clean_or_minor_difference,31,31,0,NaN,20,23.00,"10,225,339.00","10,225,339.00",0.00,"10,122,862.00",0.01
7,2018-08,clean_or_minor_difference,31,31,0,NaN,21,23.00,"10,728,463.00","10,728,463.00",0.00,"10,639,593.00",0.01
8,2018-09,clean_or_minor_difference,30,30,0,NaN,22,24.00,"9,895,337.00","9,895,337.00",0.00,"9,800,616.00",0.01
9,2018-10,clean_or_minor_difference,31,31,0,NaN,19,23.00,"11,081,974.00","11,081,974.00",0.00,"10,978,799.00",0.01


In [18]:
hourly_completeness_flag_summary_df = (
    hourly_completeness_audit_df.groupby("Quality Flag", as_index=False)
    .agg(
        periods=("Period", "count"),
        max_abs_workbook_diff_pct=("Workbook Difference Pct", lambda values: values.abs().max()),
        periods_with_missing_full_days=("Missing Full Days", lambda values: (values > 0).sum()),
        imputed_ridership_added=("Imputed Ridership Added", "sum"),
    )
    .sort_values("Quality Flag")
)

hourly_completeness_flag_summary_df


,Quality Flag,periods,max_abs_workbook_diff_pct,periods_with_missing_full_days,imputed_ridership_added
0,clean_or_minor_difference,89,0.04,0,0.00
1,complete_hours_source_discrepancy,4,0.14,0,0.00
2,missing_full_days_imputed,3,0.03,3,"1,179,297.41"


## Processed Source Coverage Audit

The app summary should be hourly-derived for every available hourly OD period. This table checks coverage after mapping current processed `Entry Station` codes through `legacy_app_code` aliases into official `station_id` values.


In [19]:
def add_station_identity(df):
    result = df.copy()
    if "station_id" not in result.columns:
        result["station_id"] = result["Entry Station"].astype(str).map(legacy_app_to_station_id)
    if "display_name" not in result.columns:
        result = result.merge(
            station_master_df[["station_id", "display_name"]],
            on="station_id",
            how="left",
            validate="many_to_one",
        )
    return result


app_keys = add_station_identity(app_summary_df[["Year", "Month", "Period", "Entry Station", "station_id", "display_name", "Source Type"]])
app_keys = app_keys[["Year", "Month", "Period", "station_id", "display_name", "Source Type"]].copy()
app_keys["In App Summary"] = True

hourly_keys = add_station_identity(hourly_summary_df[["Year", "Month", "Period", "Entry Station", "station_id", "display_name"]])
hourly_keys = hourly_keys[["Year", "Month", "Period", "station_id", "display_name"]].copy()
hourly_keys["In Hourly Summary"] = True

# Only nonzero benchmark rows are used for coverage-hole detection.
# Some workbooks include unopened future stations as explicit zero rows.
workbook_keys = add_station_identity(
    validation_audit_df[
        (validation_audit_df["Workbook Ridership"] > 0)
        | (validation_audit_df["Hourly OD Ridership"] > 0)
    ][["Year", "Month", "Period", "Entry Station", "station_id", "display_name"]]
)
workbook_keys = workbook_keys[["Year", "Month", "Period", "station_id", "display_name"]].copy()
workbook_keys["In Workbook Benchmark"] = True

coverage_audit_df = (
    app_keys
    .merge(hourly_keys, on=["Year", "Month", "Period", "station_id", "display_name"], how="outer")
    .merge(workbook_keys, on=["Year", "Month", "Period", "station_id", "display_name"], how="outer")
)
for column in ["In App Summary", "In Hourly Summary", "In Workbook Benchmark"]:
    coverage_audit_df[column] = coverage_audit_df[column].fillna(False)


def coverage_flag(row):
    if pd.isna(row["station_id"]):
        return "Missing processed station_id"
    if pd.isna(row["display_name"]):
        return "Missing processed display_name"
    if not row["In App Summary"]:
        return "Missing from app summary"
    if not row["In Hourly Summary"]:
        return "Missing from hourly summary"
    if not row["In Workbook Benchmark"]:
        return "No workbook benchmark row"
    return "Covered"


coverage_audit_df["Coverage Flag"] = coverage_audit_df.apply(coverage_flag, axis=1)
coverage_display_df = coverage_audit_df[
    [
        "Period",
        "station_id",
        "display_name",
        "Source Type",
        "In App Summary",
        "In Hourly Summary",
        "In Workbook Benchmark",
        "Coverage Flag",
    ]
]

coverage_display_df.sort_values(["Coverage Flag", "Period", "station_id"])


,Period,station_id,display_name,Source Type,In App Summary,In Hourly Summary,In Workbook Benchmark,Coverage Flag
46,2018-02,12th,12th St/Oakland City Center,hourly_od,True,True,True,Covered
47,2018-02,16th,16th St/Mission,hourly_od,True,True,True,Covered
48,2018-02,19th,19th St/Oakland,hourly_od,True,True,True,Covered
49,2018-02,24th,24th St/Mission,hourly_od,True,True,True,Covered
50,2018-02,ashb,Ashby,hourly_od,True,True,True,Covered
...,...,...,...,...,...,...,...,...
41,2018-01,ucty,Union City,hourly_od,True,True,False,No workbook benchmark row
42,2018-01,warm,Warm Springs/South Fremont,hourly_od,True,True,False,No workbook benchmark row
43,2018-01,wcrk,Walnut Creek,hourly_od,True,True,False,No workbook benchmark row
44,2018-01,wdub,West Dublin/Pleasanton,hourly_od,True,True,False,No workbook benchmark row


In [20]:
coverage_holes_df = coverage_audit_df[coverage_audit_df["Coverage Flag"] != "Covered"].copy()
coverage_holes_display_df = coverage_holes_df[
    [
        "Period",
        "station_id",
        "display_name",
        "Source Type",
        "In App Summary",
        "In Hourly Summary",
        "In Workbook Benchmark",
        "Coverage Flag",
    ]
]

coverage_holes_display_df.sort_values(["Coverage Flag", "Period", "station_id"])


,Period,station_id,display_name,Source Type,In App Summary,In Hourly Summary,In Workbook Benchmark,Coverage Flag
0,2018-01,12th,12th St/Oakland City Center,hourly_od,True,True,False,No workbook benchmark row
1,2018-01,16th,16th St/Mission,hourly_od,True,True,False,No workbook benchmark row
2,2018-01,19th,19th St/Oakland,hourly_od,True,True,False,No workbook benchmark row
3,2018-01,24th,24th St/Mission,hourly_od,True,True,False,No workbook benchmark row
4,2018-01,ashb,Ashby,hourly_od,True,True,False,No workbook benchmark row
5,2018-01,balb,Balboa Park,hourly_od,True,True,False,No workbook benchmark row
6,2018-01,bayf,Bay Fair,hourly_od,True,True,False,No workbook benchmark row
7,2018-01,cast,Castro Valley,hourly_od,True,True,False,No workbook benchmark row
8,2018-01,civc,Civic Center/UN Plaza,hourly_od,True,True,False,No workbook benchmark row
9,2018-01,colm,Colma,hourly_od,True,True,False,No workbook benchmark row


## Audit Checklist

Use this checklist before changing the app's source-of-truth rules or adding Parquet/time-lapse features.

- `stations_master_df["station_id"]` should be unique and non-null.
- `alias_target_holes_df` should be empty.
- `hourly_code_coverage_df["Unmapped Raw Codes"]` should be empty for every year.
- `hourly_completeness_display_df` should explicitly identify missing full days that were imputed and major source-definition discrepancies where hourly OD remains canonical.
- `coverage_holes_df` should contain only expected benchmark gaps, especially Jan 2018 with no comparable workbook Total Trips sheet.
- `priority_validation_rows_df` should be reviewed and categorized before claiming workbook/hourly agreement.
- Periods with large total percent differences should be investigated separately from station-code mapping issues.
